# Evaluation With Calibration Methods

## Install required dependancies

In [1]:

!pip install betacal --quiet

### Import required modules

In [2]:

import numpy as np
import pandas as pd
import pickle
import warnings
import math
import gc
import time
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# LightGBM — gradient boosted tree model
import lightgbm as lgb

# TensorFlow/Keras — LSTM model
import tensorflow as tf
from tensorflow import keras
import xarray as xr

# Scikit-learn metrics — same as original compute_metrics
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    classification_report,
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import RobustScaler

# PyTorch — needed only for temperature scaling optimisation
# Temperature scaling uses LBFGS which converges faster in PyTorch
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')

from google.colab import drive

# Beta calibration — best for LightGBM (see report)
try:
    from betacal import BetaCalibration
    BETA_AVAILABLE = True
    print('BetaCalibration available')
except ImportError:
    BETA_AVAILABLE = False
    print('betacal not installed')
    print('   Run: !pip install betacal')
    print('   Falling back to Isotonic regression')

print('\nAll imports done')



BetaCalibration available

All imports done


## Mounting google drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


## Define configurations

In [4]:

# Directory paths
BASE_DIR   = Path('/content/drive/MyDrive/MSc-research-code/')
# LightGBM models: trained_models/lgbm/{layer_folder}/
LGBM_BASE  = BASE_DIR / 'trained_models/lightgbm/'
# LSTM models: trained_models/lstm/{layer_folder}/
LSTM_BASE  = BASE_DIR / 'trained_models/lstm/'
DATA_DIR   = BASE_DIR / 'data/'
OUTPUT_DIR = BASE_DIR / 'evaluation_results/'
OUTPUT_DIR.mkdir(exist_ok=True)

# Lead times
LEAD_TIMES = [3, 5, 7, 9, 12, 14]

#  LSTM window size
WINDOW_SIZE = 15

# LSTM sentinel value
# -10.0 is outside RobustScaler range [-3,+3]
# LSTM learns "-10.0 = this depth layer absent here"
MASK_VALUE = -10.0

# Number of lag features for LightGBM
N_LAGS = 14

BATCH_SIZE = 2048

CONFIGS = {

    'Sub-SST': {
        'lgbm_dir':    LGBM_BASE / 'lightbgm_sst_subsurface_combined',
        'lstm_dir':    LSTM_BASE / 'lstm_sst_subsurface_combined',
        'lgbm_config': 'sst_subsurface_combined',
        'lstm_config': 'sst_subsurface_combined',
        'target':      'mhw_sst',
        'features':    [
            'sst', 'wind_speed', 'ssr_flux', 'str_flux',
            'latent_flux', 'sensible_flux', 't2m', 'msl',
            'temp_0_50m', 'temp_50_100m', 'temp_100_150m'
        ],
    },

    '0-50m': {
        # predicts MHW in 0-50m mixed layer
        # target: mhw_layer1
        'lgbm_dir':    LGBM_BASE / 'lightbgm_subsurface_layer_0-50m_layer',
        'lstm_dir':    LSTM_BASE / 'lstm_subsurface_layer_0-50m_layer',
        'lgbm_config': 'subsurface_layer_0-50m_layer',
        'lstm_config': 'subsurface_layer_0-50m_layer',
        'target':      'mhw_layer1',
        'features':    [
            'sst', 'wind_speed', 'ssr_flux', 'str_flux',
            'latent_flux', 'sensible_flux', 't2m', 'msl',
            'temp_0_50m', 'temp_50_100m', 'temp_100_150m'
        ],
    },

    '50-100m': {
        # predicts MHW in 50-100m thermocline layer
        # target: mhw_layer2
        'lgbm_dir':    LGBM_BASE / 'lightbgm_subsurface_layer_50-100m_layer',
        'lstm_dir':    LSTM_BASE / 'lstm_subsurface_layer_50-100m_layer',
        'lgbm_config': 'subsurface_layer_50-100m_layer',
        'lstm_config': 'subsurface_layer_50-100m_layer',
        'target':      'mhw_layer2',
        'features':    [
            'sst', 'wind_speed', 'ssr_flux', 'str_flux',
            'latent_flux', 'sensible_flux', 't2m', 'msl',
            'temp_0_50m', 'temp_50_100m', 'temp_100_150m'
        ],
    },

    '100-150m': {
        #  predicts MHW in 100-150m deep layer
        # target: mhw_layer3
        'lgbm_dir':    LGBM_BASE / 'lightbgm_subsurface_layer_100-150m_layer',
        'lstm_dir':    LSTM_BASE / 'lstm_subsurface_layer_100-150m_layer',
        'lgbm_config': 'subsurface_layer_100-150m_layer',
        'lstm_config': 'subsurface_layer_100-150m_layer',
        'target':      'mhw_layer3',
        'features':    [
            'sst', 'wind_speed', 'ssr_flux', 'str_flux',
            'latent_flux', 'sensible_flux', 't2m', 'msl',
            'temp_0_50m', 'temp_50_100m', 'temp_100_150m'
        ],
    },
}

# All features used across models
ALL_FEATURES = [
    'sst', 'wind_speed', 'ssr_flux', 'str_flux',
    'latent_flux', 'sensible_flux', 't2m', 'msl',
    'temp_0_50m', 'temp_50_100m', 'temp_100_150m'
]
ALL_TARGETS = ['mhw_sst', 'mhw_layer1', 'mhw_layer2', 'mhw_layer3']

print('Constants configured')
print(f'  Configs: {list(CONFIGS.keys())}')
print(f'  Lead times: {LEAD_TIMES}')



Constants configured
  Configs: ['Sub-SST', '0-50m', '50-100m', '100-150m']
  Lead times: [3, 5, 7, 9, 12, 14]


## Check if required models exsit in model file paths

In [5]:

print('\nVerifying model file paths...')
print('='*70)

all_found  = True
lgbm_found = 0
lgbm_miss  = 0
lstm_found = 0
lstm_miss  = 0

for config_label, cfg in CONFIGS.items():
    print(f'\n{config_label}:')
    print(f'  LightGBM folder: {cfg["lgbm_dir"]}')
    print(f'  LSTM folder:     {cfg["lstm_dir"]}')

    if not cfg['lgbm_dir'].exists():
        print(f'  \u274c LightGBM folder NOT FOUND on Drive')
        all_found = False
    if not cfg['lstm_dir'].exists():
        print(f'  \u274c LSTM folder NOT FOUND on Drive')
        all_found = False

    for lead in LEAD_TIMES:
        lgbm_path    = cfg['lgbm_dir'] / \
            f'lgbm_{cfg["lgbm_config"]}_lead{lead}.pkl'
        lstm_model   = cfg['lstm_dir'] / \
            f'lstm_{cfg["lstm_config"]}_lead{lead}_model.keras'
        lstm_scaler  = cfg['lstm_dir'] / \
            f'lstm_{cfg["lstm_config"]}_lead{lead}_scaler.pkl'
        lstm_history = cfg['lstm_dir'] / \
            f'lstm_{cfg["lstm_config"]}_lead{lead}_history.pkl'

        lgbm_ok  = lgbm_path.exists()
        lstm_ok  = lstm_model.exists() and lstm_scaler.exists()
        hist_ok  = lstm_history.exists()

        lgbm_s = '\u2705' if lgbm_ok else '\u274c'
        lstm_s = '\u2705' if lstm_ok  else '\u274c'
        hist_s = '\u2705' if hist_ok  else '\u26a0\ufe0f missing'

        print(f'  Lead {lead:2d}d:  '
              f'LightGBM {lgbm_s}  '
              f'LSTM model+scaler {lstm_s}  '
              f'history {hist_s}')

        if lgbm_ok: lgbm_found += 1
        else:       lgbm_miss += 1; all_found = False
        if lstm_ok: lstm_found += 1
        else:       lstm_miss += 1; all_found = False

print(f'\n{"─"*70}')
print(f'LightGBM: {lgbm_found} found, {lgbm_miss} missing')
print(f'LSTM:     {lstm_found} found, {lstm_miss} missing')
if all_found:
    print('\n\u2705 All model files found — ready to evaluate')
else:
    print('\n\u274c Some files missing — update folder names in CONFIGS')



Verifying model file paths...

Sub-SST:
  LightGBM folder: /content/drive/MyDrive/MSc-research-code/trained_models/lightgbm/lightbgm_sst_subsurface_combined
  LSTM folder:     /content/drive/MyDrive/MSc-research-code/trained_models/lstm/lstm_sst_subsurface_combined
  Lead  3d:  LightGBM ✅  LSTM model+scaler ✅  history ✅
  Lead  5d:  LightGBM ✅  LSTM model+scaler ✅  history ✅
  Lead  7d:  LightGBM ✅  LSTM model+scaler ✅  history ✅
  Lead  9d:  LightGBM ✅  LSTM model+scaler ✅  history ✅
  Lead 12d:  LightGBM ✅  LSTM model+scaler ✅  history ✅
  Lead 14d:  LightGBM ✅  LSTM model+scaler ✅  history ✅

0-50m:
  LightGBM folder: /content/drive/MyDrive/MSc-research-code/trained_models/lightgbm/lightbgm_subsurface_layer_0-50m_layer
  LSTM folder:     /content/drive/MyDrive/MSc-research-code/trained_models/lstm/lstm_subsurface_layer_0-50m_layer
  Lead  3d:  LightGBM ✅  LSTM model+scaler ✅  history ✅
  Lead  5d:  LightGBM ✅  LSTM model+scaler ✅  history ✅
  Lead  7d:  LightGBM ✅  LSTM model+scale

## Define required helper & evaluation metrics methods

In [ ]:


def compute_pi_rates(df_train, df_cal, df_thr, df_test,
                     target_col):
    """
    Compute actual MHW rates per split for this config+lead.

    WHY DYNAMIC (not hardcoded):
      Each depth layer has a different MHW occurrence rate.
      Each lead time drops different rows (last N rows per location).
      So rates differ per config AND per lead.

      Rates therefore differ by layer, lead time and split, so
      they are computed for every configuration rather than
      hardcoded. The SLD prior correction uses pi_cal (source) and
      pi_thr (target); pi_test is recorded for reporting only and
      is never used for fitting, selection or threshold tuning.

    Parameters
    ----------
    df_train, df_cal, df_thr, df_test : DataFrames
        Each split with target_col already present
    target_col : str  e.g. 'mhw_sst_lead3'

    Returns
    -------
    (pi_train, pi_cal, pi_thr, pi_test) : tuple of floats
    """
    def rate(df, col):
        vals = df[col].dropna().values
        return float(vals.mean()) if len(vals) > 0 else 0.0

    return (
        rate(df_train, target_col),
        rate(df_cal,   target_col),
        rate(df_thr,   target_col),
        rate(df_test,  target_col),
    )


def sedi_score(tp, fp, fn, tn):
    """
    Symmetric Extremal Dependence Index (SEDI).

    WHY USE SEDI:
      F1 and accuracy are affected by class imbalance.
      SEDI is specifically designed for rare/extreme events
      like marine heatwaves. It is invariant to the base rate
      of the event, making it ideal for comparing models across
      different depth layers that have different MHW rates.

      Range: -1 to +1
        SEDI = 1.0 --> perfect detection
        SEDI = 0.0 --> no skill (random)
        SEDI < 0   --> worse than random

      Reference: Ferro & Stephenson (2011)
        "Extremal Dependence Indices"
        Quarterly Journal of the Royal Meteorological Society
    """
    eps = 1e-7   # avoid log(0)
    tpr = tp / (tp + fn) if (tp + fn) > 0 else eps
    fpr = fp / (fp + tn) if (fp + tn) > 0 else eps
    H   = np.clip(tpr, eps, 1 - eps)   # hit rate
    F   = np.clip(fpr, eps, 1 - eps)   # false alarm rate

    try:
        # SEDI formula: uses log of hit and false alarm rates
        # numerator captures the discrimination
        # denominator normalises to [-1, 1]
        numerator   = (np.log(F) - np.log(H) -
                       np.log(1-F) + np.log(1-H))
        denominator = (np.log(F) + np.log(H) +
                       np.log(1-F) + np.log(1-H))
        val = numerator / denominator
        # return 0 if formula undefined (edge cases)
        return 0.0 if (np.isnan(val) or np.isinf(val)) \
               else float(val)
    except Exception:
        return 0.0


def brier_skill_score(y_true, y_prob):
    """
    Brier Skill Score (BSS).

    WHY USE BSS:
      Brier score alone tells you the absolute error in probabilities.
      BSS tells you how much BETTER the model is than simply
      predicting the observed MHW rate every time (climatology).

      BSS = 1 - (Brier_model / Brier_climatology)

      BSS = 1.0  --> perfect predictions
      BSS > 0    --> better than climatology
      BSS = 0    --> same skill as climatology
      BSS < 0    --> worse than climatology

      Climatology baseline:
        Always predict p = observed MHW rate
        Brier_clim = p_obs * (1 - p_obs)

      Critical for thesis: shows models add value beyond
      just knowing the base rate of MHW events.
    """
    p_clim      = float(y_true.mean())
    brier_clim  = p_clim * (1 - p_clim)  # baseline
    brier_model = float(brier_score_loss(y_true, y_prob))

    if brier_clim == 0:
        return 0.0   # avoid division by zero (degenerate case)
    return float(1.0 - brier_model / brier_clim)


def expected_calibration_error(y_true, y_prob, n_bins=15):
    """
    Expected Calibration Error (ECE) with adaptive binning.

    WHY USE ECE:
      Measures whether predicted probabilities match observed
      event frequencies. A perfectly calibrated model would
      have ECE = 0, meaning that when the model says 60%,
      MHWs occur 60% of the time.

      Using adaptive (quantile) bins rather than equal-width
      bins reduces sensitivity to the bin boundaries and avoids
      empty bins in sparse regions of probability space.

      Lower ECE = better calibration.

    Reference: Guo et al. (2017) "On Calibration of Modern
    Neural Networks"
    """
    # Quantile bins ensure each bin has equal number of points
    # This is more robust than equal-width bins
    bins = np.quantile(y_prob, np.linspace(0, 1, n_bins + 1))
    bins = np.unique(bins)   # remove duplicate bin edges

    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        # Weighted absolute difference: bin confidence vs bin accuracy
        ece += mask.mean() * abs(y_prob[mask].mean() -
                                  y_true[mask].mean())
    return float(ece)


def compute_metrics(y_test, y_pred, y_prob,
                    threshold, strategy,
                    model_name, config, target, lead,
                    calibration='none',
                    calibrated=False,
                    best_epoch=None,
                    best_val_auc=None,
                    brier_raw=None,
                    brier_calibrated=None,
                    pi_train=None,
                    pi_cal=None,
                    pi_thr=None,
                    pi_test=None):
    """
    Complete metrics function.

    IDENTICAL to original per-model compute_metrics
    (same columns in CSV for direct comparison with training results)
    PLUS new metrics and fields.

    Parameters
    ----------
    y_test    : np.ndarray — true binary labels
    y_pred    : np.ndarray — predicted binary labels (at threshold)
    y_prob    : np.ndarray — predicted probabilities
    threshold : float      — decision threshold used
    strategy  : str        — 'mhw_f1' or 'macro_f1'
    model_name: str        — 'LightGBM' or 'LSTM'
    config    : str        — e.g. 'SST Only', '0-50m'
    target    : str        — e.g. 'mhw_sst'
    lead      : int        — lead time in days
    calibration     : str  — 'none', 'Beta', 'TempScale',
                              'Beta+SLD', 'TempScale+SLD'
    calibrated      : bool — True if any calibration applied
    best_epoch      : int  — from training history
    best_val_auc    : float— from training history
    brier_raw       : float— Brier on cal split before calibration
    brier_calibrated: float— Brier on cal split after calibration
    pi_train/cal/thr/test : float — actual MHW rates per split
                              computed dynamically per config+lead
    """
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Pre-compute rates (used multiple times)
    tpr = float(tp / (tp+fn)) if (tp+fn) > 0 else 0.0
    fpr = float(fp / (fp+tn)) if (fp+tn) > 0 else 0.0
    tnr = float(tn / (tn+fp)) if (tn+fp) > 0 else 0.0
    fnr = float(fn / (tp+fn)) if (tp+fn) > 0 else 0.0
    csi = float(tp / (tp+fp+fn)) if (tp+fp+fn) > 0 else 0.0

    return {
        # Metadata
        'model':              model_name,
        'config':             config,
        'target':             target,
        'lead_time':          lead,
        'calibration':        calibration,
        'calibrated':         calibrated,
        'threshold_strategy': strategy,
        'best_threshold':     float(threshold),

        #  Training history
        # best_epoch: which epoch had best val_auc (LSTM only)
        # best_val_auc: best validation AUC during training (LSTM)
        'best_epoch':         best_epoch,
        'best_val_auc':       best_val_auc,

        #  Calibration quality on CAL split
        # Compare raw vs calibrated Brier on the CAL split
        # If brier_calibrated > brier_raw → calibration HURT
        # This is stored for each model so you can decide
        # whether to use calibration in the dashboard
        'brier_raw':          brier_raw,
        'brier_calibrated':   brier_calibrated,

        #  MHW rates per split (dynamic per config+lead)
        # pi_train: what model was trained on
        # pi_cal:   what calibrator was fitted on
        # pi_thr:   what threshold was tuned on
        # pi_test:  actual test distribution
        # These differ per config+lead — stored for traceability
        # SLD correction (after review): logit += log-odds(pi_thr) - log-odds(pi_cal)
        'pi_train':           pi_train,
        'pi_cal':             pi_cal,
        'pi_thr':             pi_thr,
        'pi_test':            pi_test,

        #  Discrimination metrics
        # AUC: ranking ability — NOT affected by calibration
        #   (monotone calibrators preserve rank order exactly)
        #   Best for comparing discrimination across models
        'auc':                float(roc_auc_score(y_test, y_prob)),

        # avg_precision: area under precision-recall curve
        #   Better than AUC-ROC for very imbalanced problems
        #   Emphasises minority class (MHW) performance
        'avg_precision':      float(average_precision_score(
                                    y_test, y_prob)),

        # Calibration quality on TEST split
        # brier: mean squared error of probabilities
        #   Combines calibration AND discrimination
        #   Lower is better
        'brier':              float(brier_score_loss(y_test, y_prob)),

        # bss: skill relative to climatological baseline
        #   BSS > 0 → better than always predicting MHW rate
        #   Essential for showing model adds value
        'bss':                brier_skill_score(y_test, y_prob),

        # log_loss: logarithmic scoring rule
        # Penalises confident wrong predictions heavily
        'log_loss':           float(log_loss(y_test, y_prob)),

        # ece: reliability of probability estimates
        # Lower = probabilities match observed frequencies better
        'ece':                expected_calibration_error(
                                    y_test, y_prob),

        # Classification metrics
        # f1: harmonic mean of precision and recall (MHW class)
        'f1':                 float(f1_score(y_test, y_pred,
                                    zero_division=0)),
        'precision':          float(precision_score(y_test, y_pred,
                                    zero_division=0)),
        'recall':             float(recall_score(y_test, y_pred,
                                    zero_division=0)),

        # f1_0, precision_0, recall_0: metrics for non-MHW class
        'f1_0':               float(f1_score(y_test, y_pred,
                                    pos_label=0, zero_division=0)),
        'precision_0':        float(precision_score(y_test, y_pred,
                                    pos_label=0, zero_division=0)),
        'recall_0':           float(recall_score(y_test, y_pred,
                                    pos_label=0, zero_division=0)),

        'accuracy':           float(accuracy_score(y_test, y_pred)),

        # macro_f1: average F1 across both classes
        'macro_f1':           float(f1_score(y_test, y_pred,
                                    average='macro', zero_division=0)),

        # Confusion matrix raw counts
        'tn': int(tn), 'fp': int(fp),
        'fn': int(fn), 'tp': int(tp),

        # Hit/miss rates
        # tpr: true positive rate = hit rate = recall
        # Fraction of actual MHWs that were detected
        'tpr': tpr,

        # tnr: true negative rate = specificity
        'tnr': tnr,

        # fpr: false positive rate = false alarm rate
        # Fraction of actual non-MHWs incorrectly flagged
        'fpr': fpr,

        # fnr: false negative rate = miss rate
        'fnr': fnr,

        # csi: critical success index = threat score
        #  Standard meteorological verification metric
        'csi': csi,

        # sedi: Symmetric Extremal Dependence Index
        # Best metric for rare event detection
        #  NOT affected by class imbalance like F1
        'sedi': sedi_score(tp, fp, fn, tn),

        # Shows what the model actually outputs
        # prob_mean vs pi_test shows miscalibration direction:
        #   prob_mean << pi_test --> model under-predicts MHW
        #   prob_mean >> pi_test --> model over-predicts MHW
        'prob_mean':   float(y_prob.mean()),
        'prob_std':    float(y_prob.std()),
        'prob_median': float(np.median(y_prob)),
        'prob_pct10':  float(np.percentile(y_prob, 10)),
        'prob_pct90':  float(np.percentile(y_prob, 90)),

        #  Dataset info
        'n_test':        int(len(y_test)),
        'n_pos_test':    int(y_test.sum()),
        'n_neg_test':    int((1 - y_test).sum()),
        'mhw_rate_test': float(y_test.mean()),
    }


def tune_thresholds(y_true, y_prob):
    """
    Tune decision threshold on a held-out split.

    WHY TWO STRATEGIES:
      mhw_f1:   maximises detection of MHW events
                lower threshold → more detections
                higher recall, possibly more false alarms
                best for early warning where misses are costly

      macro_f1: balances detection of both MHW and non-MHW
                higher threshold → fewer false alarms
                better balance between classes
                best for operational efficiency

    CRITICAL: threshold always tuned on THR split
    NOT on CAL split (already used for calibration fitting)
    NOT on TEST split (never used for tuning)
    Using same set for both calibration AND threshold = double-dipping --> biases F1 estimates optimistically
    """
    thresholds = np.arange(0.01, 0.99, 0.01)

    # Strategy 1: maximise F1 for MHW class
    f1_mhw_scores = [
        f1_score(y_true, (y_prob > t).astype(int), zero_division=0)
        for t in thresholds
    ]
    best_mhw = float(thresholds[np.argmax(f1_mhw_scores)])

    # Strategy 2: maximise macro (average) F1
    f1_mac_scores = [
        f1_score(y_true, (y_prob > t).astype(int),
                 average='macro', zero_division=0)
        for t in thresholds
    ]
    best_mac = float(thresholds[np.argmax(f1_mac_scores)])

    return best_mhw, best_mac


print('Metric functions defined')
print('  compute_pi_rates()           — dynamic MHW rates per split')
print('  sedi_score()                 — extreme event metric')
print('  brier_skill_score()          — skill vs climatology')
print('  expected_calibration_error() — probability reliability')
print('  compute_metrics()            — full metric suite')
print('  tune_thresholds()            — two-strategy threshold')


# =====================================================================
# ADDED AFTER PEER REVIEW
#   1. Variant-selection scores on the THR split (never on test data)
#   2. Persistence baseline
#   3. Onset recall (new events only)
# =====================================================================

def selection_scores(y_thr, variant_probs_thr):
    """
    Brier Score and ECE of each calibration variant on the THR split.

    These scores are used ONLY to choose the calibration variant.
    The THR split (2021-01-31 to 2022-12-31) is disjoint from the CAL
    split (used to fit the calibrator) and from the TEST split (used
    only for reported metrics), so the choice uses no test labels.
    """
    return {
        name: (float(brier_score_loss(y_thr, p)),
               float(expected_calibration_error(y_thr, p)))
        for name, p in variant_probs_thr.items()
    }


def add_selection_and_onset(rows, sel_scores, variant_probs_test,
                            y_test, y_now):
    """
    Attach to every evaluation row:
      brier_thr, ece_thr : THR-split scores of its variant (for selection)
      n_onset, onset_tpr : recall on NEW events only, i.e. test samples
                           with no MHW on day t and an MHW on day t+lead.
    """
    y_test = np.asarray(y_test).astype(np.int8)
    y_now  = np.asarray(y_now).astype(np.int8)
    onset  = (y_now == 0) & (y_test == 1)
    for row in rows:
        b_thr, e_thr = sel_scores[row['calibration']]
        row['brier_thr'] = b_thr
        row['ece_thr']   = e_thr
        probs  = variant_probs_test[row['calibration']]
        y_pred = (probs > row['best_threshold']).astype(int)
        row['n_onset']   = int(onset.sum())
        row['onset_tpr'] = (float(y_pred[onset].mean())
                            if onset.any() else np.nan)
    return rows


def persistence_row(y_test, y_now, model_rows_label,
                    config_label, target, lead,
                    pi_train=None, pi_cal=None,
                    pi_thr=None, pi_test=None):
    """
    Persistence baseline: the MHW state on day t is the forecast for
    day t+lead. Scored with compute_metrics(), exactly like the models,
    on the SAME test rows as the model it is compared with.

    The forecast is binary, so probabilities are 0/1 (clipped by 1e-6
    only to keep log-loss finite; Brier changes by < 1e-11).
    """
    y_test = np.asarray(y_test).astype(np.int8)
    y_now  = np.asarray(y_now).astype(np.int8)
    probs  = np.clip(y_now.astype(float), 1e-6, 1 - 1e-6)
    row = compute_metrics(
        y_test, y_now.astype(int), probs,
        0.5, 'persistence',
        f'Persistence ({model_rows_label} rows)',
        config_label, target, lead,
        calibration='persistence', calibrated=False,
        pi_train=pi_train, pi_cal=pi_cal,
        pi_thr=pi_thr, pi_test=pi_test,
    )
    onset = (y_now == 0) & (y_test == 1)
    row['n_onset']   = int(onset.sum())
    row['onset_tpr'] = 0.0 if onset.any() else np.nan   # misses every onset by definition
    row['brier_thr'] = np.nan
    row['ece_thr']   = np.nan
    return row


def lstm_state_at_t(df, target, ds):
    """
    MHW state on day t for every LSTM test sequence, in the same order
    as ds.get_labels() and model.predict().
    MaskedLSTMDataset sorts df by (location, time) and stores target_idx
    into that sorted frame, so the same sort is reproduced here.
    """
    df_sorted = df.sort_values(['location', 'time']).reset_index(drop=True)
    now = np.nan_to_num(df_sorted[target].values.astype(float), nan=0.0)
    return np.array([now[ti] for _, ti in ds.sequence_indices]).astype(np.int8)


print('Added: selection_scores(), add_selection_and_onset(), '
      'persistence_row(), lstm_state_at_t()')


# =====================================================================
# ADDED AFTER PEER REVIEW (2): common cells + saved test predictions
# =====================================================================

def lstm_cols_at_t(df, cols, ds):
    """
    Values of df columns on day t for every LSTM test sequence, in the
    same order as ds.get_labels() / model.predict(). Reproduces the
    (location, time) sort used inside MaskedLSTMDataset.
    """
    df_sorted = df.sort_values(['location', 'time']).reset_index(drop=True)
    idx = np.fromiter((ti for _, ti in ds.sequence_indices),
                      dtype=np.int64, count=len(ds.sequence_indices))
    return {c: df_sorted[c].values[idx] for c in cols}


def common_cell_rows(rows, variant_probs_test, y_test, y_now, mask,
                     pis_common, model_name, config_label, target, lead):
    """
    Re-score the SAME predictions (same variants, same thresholds) on the
    common cells only: cells with valid 100-150 m data, i.e. seafloor
    deeper than 150 m. Only the MHW-F1 rows are re-scored. Adds the
    persistence baseline on the same common rows.
    """
    y_test = np.asarray(y_test).astype(np.int8)
    y_now  = np.asarray(y_now).astype(np.int8)
    yt, yn = y_test[mask], y_now[mask]
    out = []
    if mask.sum() == 0 or yt.min() == yt.max():
        return out
    for row in rows:
        if row['threshold_strategy'] != 'mhw_f1':
            continue
        p = variant_probs_test[row['calibration']][mask]
        r = compute_metrics(
            yt, (p > row['best_threshold']).astype(int), p,
            row['best_threshold'], 'mhw_f1',
            model_name, config_label, target, lead,
            calibration=row['calibration'], calibrated=row['calibrated'],
            **pis_common)
        r['subset'] = 'common_cells'
        out.append(r)
    pr = persistence_row(yt, yn, model_name, config_label, target, lead,
                         **pis_common)
    pr['subset'] = 'common_cells'
    out.append(pr)
    return out


def save_test_predictions(path, locations, times, y_test, y_now,
                          variant_probs_test):
    """
    Save per-sample test predictions.
    One file per model x config x lead (~20 MB each).
    """
    df_p = pd.DataFrame({
        'location': pd.Categorical(np.asarray(locations)),
        'time':     pd.to_datetime(np.asarray(times)),
        'y':        np.asarray(y_test).astype(np.int8),
        'y_now':    np.asarray(y_now).astype(np.int8),
    })
    for name, p in variant_probs_test.items():
        df_p[f'p__{name}'] = np.asarray(p).astype(np.float32)
    df_p.to_parquet(path, index=False)


print('Added: lstm_cols_at_t(), common_cell_rows(), save_test_predictions()')


Metric functions defined
  compute_pi_rates()           — dynamic MHW rates per split
  sedi_score()                 — extreme event metric
  brier_skill_score()          — skill vs climatology
  expected_calibration_error() — probability reliability
  compute_metrics()            — full metric suite
  tune_thresholds()            — two-strategy threshold
Added: selection_scores(), add_selection_and_onset(), persistence_row(), lstm_state_at_t()
Added: lstm_cols_at_t(), common_cell_rows(), save_test_predictions()


## Define required calibation methods

In [ ]:

class TemperatureScaler(nn.Module):
    """
    Single-parameter temperature scaling for LSTM.

    HOW IT WORKS:
      The LSTM outputs a logit z (pre-sigmoid value).
      divide by temperature T: z_scaled = z / T

      If T > 1 --> logits shrunk --> probs pulled toward 0.5
                --> corrects UNDER-confidence from focal loss

      If T < 1 --> logits expanded --> probs pushed to 0/1
                --> corrects over-confidence

      T is optimised by minimising NLL on the calibration
      split using LBFGS.

      KEY ADVANTAGE: only 1 parameter --> cannot overfit
      KEY ADVANTAGE: preserves AUC and ranking exactly
      (monotone transformation of logits)
    """
    def __init__(self):
        super().__init__()
        # Initialise T=1.0 (no scaling) via log parameterisation
        # log_T ensures T > 0 always (unconstrained optimisation)
        self.log_T = nn.Parameter(torch.zeros(1))

    @property
    def T(self):
        return torch.exp(self.log_T).item()

    def forward(self, logits):
        # Scale logits by temperature T
        return logits / torch.exp(self.log_T)

    def fit(self, logits_np, y_np, lr=0.01, max_iter=500):
        """
        Optimise T by minimising binary cross-entropy on logits.
        LBFGS used (not SGD) because:
          - Only 1 parameter to optimise
          - LBFGS converges much faster for low-dimensional problems
          - Better numerical stability
        """
        bce    = nn.BCEWithLogitsLoss()
        opt    = torch.optim.LBFGS(
            [self.log_T], lr=lr, max_iter=max_iter
        )
        logits = torch.tensor(logits_np, dtype=torch.float32)
        y      = torch.tensor(y_np,      dtype=torch.float32)

        def closure():
            opt.zero_grad()
            loss = bce(self.forward(logits), y)
            loss.backward()
            return loss

        opt.step(closure)
        return self

    def predict_proba(self, logits_np):
        """Apply temperature scaling and sigmoid to get probs."""
        logits = torch.tensor(logits_np, dtype=torch.float32)
        with torch.no_grad():
            scaled = self.forward(logits)
        return torch.sigmoid(scaled).numpy()


def prob_to_logit(p, eps=1e-6):
    """
    Extract pre-sigmoid logits from LSTM.

    WHY LOGITS NOT PROBS:
      Temperature scaling works on the logit space (pre-sigmoid).
      If we used sigmoid outputs we would be doing Platt scaling
      (2-parameter method, less principled for focal loss).
      Temperature scaling on logits is theoretically optimal for
      models trained with focal loss.

    Detects whether last layer has sigmoid and goes one step back.
    """
    """Inverse sigmoid — converts probability to logit"""
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def fit_lgbm_calibrator(raw_probs_cal, y_cal):
    """
    Fit Beta calibration for LightGBM.

    WHY BETA NOT PLATT:
      Platt scaling (sigmoid) assumes class-conditional score
      distributions are Gaussian with EQUAL variance.
      This assumption is violated when using scale_pos_weight
      on imbalanced data (your MHW rate = 9.7%).

      Beta calibration uses a more flexible parametric form that
      correctly handles asymmetric score distributions.
      It recovers the identity when already calibrated.

    Falls back to Isotonic if betacal not installed.
    Isotonic is non-parametric but needs more data (~1000+ samples).
    """
    if BETA_AVAILABLE:
        # 'abm' = full parametrisation (a, b, m parameters)
        # More flexible than 'ab' or 'a' variants
        cal = BetaCalibration(parameters='abm')
        cal.fit(raw_probs_cal, y_cal)
        method = 'Beta'
    else:
        # Isotonic regression fallback
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(raw_probs_cal, y_cal)
        method = 'Isotonic'

    return cal, method


def apply_calibrator(calibrator, probs):
    """Apply fitted calibrator to new probabilities."""
    return calibrator.predict(probs)


def sld_prior_correction(probs, pi_source, pi_target):
    """
    Saerens-Latinne-Decaestecker (SLD) label shift correction.

    WHY THIS IS NEEDED:
      model was trained on data with ~9.7% MHW rate.
      The calibrator was fitted on validation data with ~25% rate.
      The test set has a much higher MHW rate (varies per config+lead).

      Even a perfectly calibrated model for the calibration
      distribution will be WRONG on the test distribution.
      This is label shift (prior shift): p(y) changed,
      but p(x|y) is approximately constant.

    HOW IT WORKS:
      Bayes-rule closed form in logit space:
        logit_corrected = logit_calibrated
                        + log(pi_target) - log(1-pi_target)
                        - log(pi_source) + log(1-pi_source)

      This is EXACT under pure label shift.
      It REQUIRES the source probs to be calibrated first.
      (That is why we calibrate BEFORE applying SLD.)

    KEY: pi_source and pi_target are computed DYNAMICALLY
    per config × per lead — NOT hardcoded constants.
    Each depth layer and lead time has different MHW rates.

    Parameters
    ----------
    probs     : calibrated probs for pi_source distribution
    pi_source : positive rate in calibration set
                (computed dynamically for this config+lead)
    pi_target : positive rate in the threshold-tuning (THR) split
                (computed dynamically for this config+lead)
    """
    eps    = 1e-7
    probs  = np.clip(probs, eps, 1 - eps)

    # Convert to logit space
    logits = np.log(probs / (1 - probs))

    # Shift logits by difference in log-odds of base rates
    logit_source = np.log(pi_source / (1 - pi_source))
    logit_target = np.log(pi_target / (1 - pi_target))
    shift        = logit_target - logit_source

    # Corrected probabilities
    return 1.0 / (1.0 + np.exp(-(logits + shift)))


def plot_reliability_diagram(y_true, prob_dict,
                              config, lead,
                              save_path=None):
    """
    Reliability diagram comparing calibration approaches.

    HOW TO READ:
      Perfect calibration = diagonal line
      Below diagonal = model over-confident (predicts too high)
      Above diagonal = model under-confident (predicts too low)

      Focal loss typically produces UNDER-confident models
      --> temperature scaling pulls toward diagonal

      SLD correction shifts the overall probability level
          toward the THR-split MHW rate.

    prob_dict : dict of {'label': probs_array}
      e.g. {'Raw': raw_probs, 'TempScale': ts_probs,
             'TempScale+SLD': sld_probs}
    """
    n_variants = len(prob_dict)
    fig, axes  = plt.subplots(
        1, n_variants, figsize=(5.5 * n_variants, 5)
    )
    if n_variants == 1:
        axes = [axes]

    fig.suptitle(
        f'Reliability Diagrams — {config} | Lead {lead}d\n'
        f'n_test={len(y_true):,}  '
        f'MHW rate={y_true.mean()*100:.1f}%',
        fontsize=12, fontweight='bold'
    )
    colors = ['#1a6b8a', '#e05c2a', '#2e9e6b']

    for i, (label, probs) in enumerate(prob_dict.items()):
        ax    = axes[i]
        color = colors[i % len(colors)]

        # Perfect calibration reference line
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.4,
                label='Perfect calibration')

        # Calibration curve with 15 adaptive bins
        prob_true, prob_pred = calibration_curve(
            y_true, probs, n_bins=15, strategy='quantile'
        )
        ax.plot(prob_pred, prob_true,
                marker='o', color=color,
                linewidth=2, markersize=6, label=label)
        ax.fill_between(prob_pred, prob_pred, prob_true,
                        alpha=0.08, color=color)

        # Histogram of predicted probabilities (inset)
        ax_in = ax.inset_axes([0.58, 0.05, 0.4, 0.28])
        ax_in.hist(probs, bins=20, color=color, alpha=0.6)
        ax_in.set_xlabel('Prob', fontsize=7)
        ax_in.tick_params(labelsize=6)

        brier = brier_score_loss(y_true, probs)
        ece   = expected_calibration_error(y_true, probs)
        bss_v = brier_skill_score(y_true, probs)

        ax.set_title(
            f'{label}\n'
            f'Brier={brier:.4f}  ECE={ece:.4f}  BSS={bss_v:.3f}',
            fontsize=9, fontweight='bold'
        )
        ax.set_xlabel('Mean predicted probability', fontsize=9)
        ax.set_ylabel('Fraction of positives', fontsize=9)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'    Saved: {Path(save_path).name}')
    plt.show()


print('Calibration functions defined ')
print('  TemperatureScaler        — for LSTM (focal loss)')
print('  get_lstm_logits          — extracts pre-sigmoid logits')
print('  fit_lgbm_calibrator      — Beta or Isotonic for LightGBM')
print('  sld_prior_correction     — dynamic label shift correction')
print('  plot_reliability_diagram — visual calibration check')



Calibration functions defined 
  TemperatureScaler        — for LSTM (focal loss)
  get_lstm_logits          — extracts pre-sigmoid logits
  fit_lgbm_calibrator      — Beta or Isotonic for LightGBM
  sld_prior_correction     — dynamic label shift correction
  plot_reliability_diagram — visual calibration check


## Evaluation method to evaluate each model

In [8]:

def evaluate_all_calibrations(
        # Threshold tuning split (DISJOINT from cal split!)
        y_thr, raw_thr, cal_thr, sld_thr,
        # Test split (final evaluation only)
        y_test, raw_test, cal_test, sld_test,
        # Metadata
        model_name, config_label, target, lead,
        cal_method,
        # Training info (passed through to compute_metrics)
        best_epoch=None, best_val_auc=None,
        brier_raw=None, brier_calibrated=None,
        # Dynamic PI rates (different per config+lead)
        pi_train=None, pi_cal=None,
        pi_thr=None, pi_test=None
):
    """
    Evaluate three calibration states × two threshold strategies.

    SPLIT DISCIPLINE:
      y_thr  --> used ONLY for threshold tuning
      y_test -->  used ONLY for final metrics

      NOT used y_test for threshold tuning.
      as this would be test-set leakage and inflate F1 estimates.

    Returns 6 rows:
      none        + mhw_f1   <- raw without calibration
      none        + macro_f1 <- raw without calibration
      cal         + mhw_f1   <- after Beta/TempScale
      cal         + macro_f1
      cal+SLD     + mhw_f1   <- calibrated + prior correction
      cal+SLD     + macro_f1
    """
    rows = []

    # Tune thresholds on THR split
    # Each calibration state gets its OWN threshold
    # because the probability scale changes after calibration
    thr_raw_mhw, thr_raw_mac = tune_thresholds(y_thr, raw_thr)
    thr_cal_mhw, thr_cal_mac = tune_thresholds(y_thr, cal_thr)
    thr_sld_mhw, thr_sld_mac = tune_thresholds(y_thr, sld_thr)

    print(f'\n  Thresholds (tuned on THR split — disjoint from CAL):')
    print(f'    none:    mhw_f1={thr_raw_mhw:.2f}  '
          f'macro_f1={thr_raw_mac:.2f}')
    print(f'    {cal_method}: mhw_f1={thr_cal_mhw:.2f}  '
          f'macro_f1={thr_cal_mac:.2f}')
    print(f'    +SLD:    mhw_f1={thr_sld_mhw:.2f}  '
          f'macro_f1={thr_sld_mac:.2f}')

    # Three variants: (name, test_probs, thr_mhw, thr_mac, is_cal)
    variants = [
        ('none',
         raw_test, thr_raw_mhw, thr_raw_mac, False),
        (cal_method,
         cal_test, thr_cal_mhw, thr_cal_mac, True),
        (f'{cal_method}+SLD',
         sld_test, thr_sld_mhw, thr_sld_mac, True),
    ]

    for cal_name, probs, thr_mhw, thr_mac, is_cal in variants:
        for strategy, threshold in [
            ('mhw_f1',   thr_mhw),
            ('macro_f1', thr_mac),
        ]:
            y_pred = (probs > threshold).astype(int)

            row = compute_metrics(
                y_test, y_pred, probs,
                threshold, strategy,
                model_name, config_label,
                target, lead,
                calibration      = cal_name,
                calibrated       = is_cal,
                best_epoch       = best_epoch,
                best_val_auc     = best_val_auc,
                brier_raw        = brier_raw,
                brier_calibrated = brier_calibrated,
                pi_train         = pi_train,
                pi_cal           = pi_cal,
                pi_thr           = pi_thr,
                pi_test          = pi_test,
            )
            rows.append(row)

            # Print same format as original training loop
            print(f'\n  === {cal_name.upper()} | '
                  f'{strategy.upper()} | '
                  f'threshold={threshold:.2f} ===')
            print(classification_report(
                y_test, y_pred,
                target_names=['No MHW (0)', 'MHW (1)'],
                digits=3
            ))
            cm    = confusion_matrix(y_test, y_pred)
            tn_v, fp_v, fn_v, tp_v = cm.ravel()
            print(f'  TN={tn_v:,}  FP={fp_v:,}  '
                  f'FN={fn_v:,}  TP={tp_v:,}')
            print(f'  AUC={row["auc"]:.3f}  '
                  f'F1={row["f1"]:.3f}  '
                  f'Macro-F1={row["macro_f1"]:.3f}  '
                  f'TPR={row["tpr"]:.3f}  '
                  f'FPR={row["fpr"]:.3f}')
            print(f'  SEDI={row["sedi"]:.3f}  '
                  f'BSS={row["bss"]:.3f}  '
                  f'Brier={row["brier"]:.4f}  '
                  f'ECE={row["ece"]:.4f}')
            print(f'  prob_mean={row["prob_mean"]:.3f}  '
                  f'prob_median={row["prob_median"]:.3f}  '
                  f'pi_test={pi_test:.3f}')

    return rows


print('evaluate_all_calibrations() defined')
print('  Returns 6 rows per model per lead:')
print('  3 calibration states × 2 threshold strategies')


evaluate_all_calibrations() defined
  Returns 6 rows per model per lead:
  3 calibration states × 2 threshold strategies


##  Define the method to convert dataset into tabluar format

In [9]:


# Load Data
def xr_to_df(ds, features, targets):
    """Convert xarray Dataset to pandas DataFrame."""
    print('Converting xarray → pandas...')
    stacked = ds[features + targets].stack(sample=('time', 'latitude', 'longitude'))
    df = stacked.to_dataframe().reset_index()

    initial_len = len(df)
    df = df.dropna(subset=['sst'])
    print(f'  Removed {initial_len - len(df):,} land points')

    df = df.sort_values(['latitude', 'longitude', 'time']).reset_index(drop=True)
    df['location'] = df['latitude'].astype(str) + '_' + df['longitude'].astype(str)

    for col in features:
        if col in df.columns:
            df[col] = df[col].astype(np.float32)
    for col in targets:
        if col in df.columns:
            df[col] = df[col].astype(np.int8)

    print(f'  Samples: {len(df):,}')
    return df




## Load the dataset from google drive and convert to tabular format

In [10]:
print('Loading xarray dataset...')
ds = xr.open_dataset(DATA_DIR / 'ocean_atmos_DAILY_WITH_LABELS_1993_2025.nc')

df = xr_to_df(ds, ALL_FEATURES, ALL_TARGETS)
print(f'Data loaded: {df.shape}')
print('='*80)


Loading xarray dataset...
Converting xarray → pandas...
  Removed 3,061,462 land points
  Samples: 11,703,463
Data loaded: (11703463, 21)


## Split dataset in to four datasets

In [11]:

TRAIN_END_DATE = '2019-12-31'
VAL_START_DATE = '2020-01-15'
VAL_MID    = '2021-01-30'
VAL_END_DATE = '2022-12-31'
TEST_START_DATE = '2023-01-15'

df_train_raw = df[df['time'] <= TRAIN_END_DATE].copy()
df_cal_raw   = df[(df['time'] >= VAL_START_DATE) &
               (df['time'] <= VAL_MID)].copy()   # calibration
df_thr_raw   = df[(df['time'] >  VAL_MID) &
               (df['time'] <= VAL_END_DATE)].copy()   # threshold tuning
df_test_raw  = df[df['time'] >= TEST_START_DATE].copy()   # final evaluation

print(f'\nSplits:')
print(f'  Train: {len(df_train_raw):,}  '
      f'({df_train_raw["time"].min().date()} → '
      f'{df_train_raw["time"].max().date()})')
print(f'  Cal:   {len(df_cal_raw):,}  <-  fit calibrators ')
print(f'  Thr:   {len(df_thr_raw):,}  <- tune thresholds ')
print(f'  Test:  {len(df_test_raw):,} <-  final evaluation')

del df; gc.collect()

# =====================================================================
# ADDED AFTER PEER REVIEW: common cells (seafloor deeper than 150 m)
# A cell is "common" if its 100-150 m temperature is valid on >= 99% of
# test days. Missingness is structural (bathymetry), so this is stable.
# Every depth configuration is re-scored on these cells only, so the
# depth comparison uses the same locations for all layers.
# =====================================================================
_valid = (df_test_raw['temp_100_150m'].notna()
          .groupby(df_test_raw['location']).mean())
COMMON_LOCS = set(_valid.index[_valid >= 0.99])
print(f'\nCommon cells (valid 100-150 m data): '
      f'{len(COMMON_LOCS):,} of {df_test_raw["location"].nunique():,}')
del _valid



Splits:
  Train: 9,575,031  (1993-01-01 → 2019-12-31)
  Cal:   370,922  <-  fit calibrators 
  Thr:   679,700  <- tune thresholds 
  Test:  1,050,622 <-  final evaluation

Common cells (valid 100-150 m data): 867 of 971


## Define method to create lead time labels



In [12]:

def create_multi_output_targets(df, target_col, lead_times):

    """
    Create multi-output targets for a specific MHW variable.

    Creates columns: {target_col}_lead3, _lead5, ..._lead14
    Drops rows where ANY lead target is NaN
    (last max(lead_times) rows per location cannot have all leads)

    WHY ALL LEADS AT ONCE:
      If we dropped NaN per-lead separately, each lead would have
      different rows — making cross-lead comparison inconsistent.
      Dropping where ANY lead is NaN ensures every remaining row
      has valid labels for ALL lead times simultaneously.
      This matches the training pipeline exactly

    WHY THIS MATTERS FOR EVALUATION:
      The MaskedLSTMDataset skips rows with NaN target internally.
      But for LightGBM we need consistent rows across leads.
      Using this function ensures evaluation uses the exact same
      row set as training for each target variable.
    """

    df = df.copy().sort_values(['location', 'time']).reset_index(drop=True)

    target_cols = []
    for lead in lead_times:
        col_name = f'{target_col}_lead{lead}'
        # Shift target FORWARD by lead days (negative shift)
        df[col_name] = df.groupby('location', sort=False)[target_col].shift(-lead)
        target_cols.append(col_name)

    # Remove rows with NaN in any target
    initial_len = len(df)
    df = df.dropna(subset=target_cols)
    dropped = initial_len - len(df)

    print(f'  Target: {target_col:15s} | Created: {len(target_cols)} | '
          f'Samples: {len(df):,} (dropped {dropped:,})')

    return df, target_cols

# Create targets for EACH configuration
print('Creating Multi-Output Targets')
print('='*80)



Creating Multi-Output Targets


## Define method to create lags for LightGBM

In [13]:

def create_lag_features(df, features, n_lags=14):
    """
    Create lag features for LightGBM.
    WHY LAGS:
      LightGBM is not a sequential model — it sees one row at a time.
      Lag features provide the temporal context that LSTM gets from
      its hidden state and window.

    WHY LAG0 INCLUDED (range starts at 0):
      lag0 = today's value, lag1 = yesterday, ..., lag14 = 14 days ago
      Including lag0 means LightGBM sees today + 14 days of history.
      This matches the LSTM window of 15 timesteps exactly.
      Total lag cols = n_features × (n_lags + 1)
        e.g. 11 features × 15 = 165 for depth models

    WHY ONLY ALL-NaN ROWS DROPPED:
      Coastal shallow locations may have NaN for subsurface features.
      These are valid ocean points — NaN means "no data at this depth"
      not "invalid location". LightGBM handles NaN natively
      Only rows where ALL lag features are NaN (startup rows at the
      beginning of each location's time series) are dropped.
      This matches the training pipeline exactly
    """
    df = df.copy().sort_values(['location', 'time']).reset_index(drop=True)

    lag_cols = []
    for feat in features:
        for lag in range(0, n_lags + 1):
            col_name = f'{feat}_lag{lag}'
            # Shift DOWN (into past)
            df[col_name] = df.groupby('location', sort=False)[feat].shift(lag)
            lag_cols.append(col_name)

    # Remove rows with NaN from lagging
    # df = df.dropna(subset=lag_cols)
    all_nan_mask = df[lag_cols].isna().all(axis=1)
    df = df[~all_nan_mask]
    print(f'  Dropped {all_nan_mask.sum():,} all-NaN startup rows')

    print(f'  Features: {len(features)} | Lags: {n_lags} | '
          f'Total lag cols: {len(lag_cols)} | Samples: {len(df):,}')

    return df, lag_cols

## Define method to create sequances for LSTM models

In [14]:


class MaskedLSTMDataset(tf.keras.utils.Sequence):
    """
    Memory-efficient LSTM dataset.
    IDENTICAL to training pipeline.

    KEY DESIGN DECISIONS (same as training):
      - Converts DataFrame to NumPy ONCE (not per batch)
        --> 100-1000x faster than pandas .loc per batch
      - Stores actual loc_indices not start/end slice
        -->  gap-safe even if days missing per location
      - RobustScaler handles NaN natively (NaN in → NaN out)
      - NaN positions replaced with mask_value=-10.0 AFTER scaling
        --> LSTM learns -10.0 = "this depth layer absent here"
      - Only sequences with non-NaN targets included
    """
    def __init__(self, df, features, target_col,
                 window_size, batch_size,
                 scaler=None, mask_value=-10.0,
                 shuffle=False):
        #sort by location then time for correct sequence order
        df_sorted = df.sort_values(
            ['location', 'time']
        ).reset_index(drop=True)

        print('  Converting DataFrame to NumPy...')
        #convert to NumPy ONCE — much faster than pandas .loc per batch
        self.X_data    = df_sorted[features].values.astype(np.float32)
        self.y_data    = df_sorted[target_col].values.astype(np.float32)
        self.locations = df_sorted['location'].values

        self.features    = features
        self.n_features  = len(features)
        self.target_col  = target_col
        self.window_size = window_size
        self.batch_size  = batch_size
        self.scaler      = scaler
        self.mask_value  = mask_value   # -10.0
        self.shuffle     = shuffle

        #build sequence index list
        #each entry: (loc_indices_array, target_idx)
        #storing actual indices (not start:end) --> gap-safe
        self.sequence_indices = []
        print('  Building sequence indices...')
        unique_locs    = np.unique(self.locations)
        location_count = 0

        for location in unique_locs:
            location_count += 1
            if location_count % 1000 == 0:
                print(f'    {location_count:,} locations | '
                      f'{len(self.sequence_indices):,} sequences')
            loc_mask    = self.locations == location
            loc_indices = np.where(loc_mask)[0]
            n           = len(loc_indices)
            if n <= window_size:
                continue   #not enough data for even one sequence
            for i in range(n - window_size):
                target_idx = loc_indices[i + window_size - 1]
                #skip if target label is NaN
                #(happens at end of each location due to lead shift)
                if np.isnan(self.y_data[target_idx]):
                    continue
                #store actual indices — gap safe
                self.sequence_indices.append((
                    loc_indices[i:i + window_size].copy(),
                    target_idx
                ))

        self.n_sequences = len(self.sequence_indices)

        #memory usage info
        X_mb   = self.X_data.nbytes / 1024**2
        y_mb   = self.y_data.nbytes / 1024**2
        seq_mb = (self.n_sequences * window_size * 8) / 1024**2
        print(f'  Dataset ready: {self.n_sequences:,} sequences')
        print(f'    X array:  {X_mb:.1f} MB')
        print(f'    y array:  {y_mb:.1f} MB')
        print(f'    Indices:  {seq_mb:.1f} MB')
        print(f'    Batches:  {len(self):,}')

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(self.n_sequences / self.batch_size))

    def __getitem__(self, idx):
        batch_start       = idx * self.batch_size
        batch_end         = min((idx + 1) * self.batch_size,
                                self.n_sequences)
        batch_meta        = self.sequence_indices[batch_start:batch_end]
        batch_size_actual = len(batch_meta)

        X_batch = np.empty(
            (batch_size_actual, self.window_size, self.n_features),
            dtype=np.float32
        )
        y_batch = np.empty(batch_size_actual, dtype=np.float32)

        #gap-safe indexing — uses stored loc_indices per sequence
        for i, (loc_indices, target_idx) in enumerate(batch_meta):
            X_batch[i] = self.X_data[loc_indices]
            y_batch[i] = self.y_data[target_idx]

        #NaN handling — correct order:
        # 1. Record NaN positions BEFORE anything else
        # 2. Scale (RobustScaler: NaN in --> NaN out natively)
        # 3. Apply -10.0 to NaN positions AFTER scaling
        nan_mask = np.isnan(X_batch)

        if self.scaler is not None:
            original_shape = X_batch.shape
            X_flat         = X_batch.reshape(-1, self.n_features)
            #robustScaler handles NaN natively:
            #non-NaN --> scaled to [-3,+3] range
            #NaN--> stays NaN (not touched)
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                X_flat_scaled = self.scaler.transform(X_flat)
            X_batch = X_flat_scaled.reshape(original_shape)

        #apply sentinel AFTER scaling
        #-10.0 is clearly outside RobustScaler range [-3,+3]
        X_batch[nan_mask] = self.mask_value

        y_batch = y_batch.reshape(-1, 1)
        return X_batch, y_batch

    def on_epoch_end(self):
        #shuffle sequence indices at end of each epoch if shuffle=True
        #at EVALUATION shuffle=False → order preserved
        #(get_labels() must match prediction order)
        if self.shuffle:
            np.random.shuffle(self.sequence_indices)

    def get_labels(self):
        """
        Extract true labels in sequence order.
        Used for evaluation
        Order matches model.predict() output
        """
        return np.array([
            self.y_data[ti]
            for _, ti in self.sequence_indices
        ]).astype(np.int8)

print('MaskedLSTMDataset defined')

MaskedLSTMDataset defined


## Evaluate all LIGHTGBM models

In [ ]:

all_results = []   # accumulates ALL rows across all models
common_results = []   # ADDED: re-scored rows on common cells only
PRED_DIR = OUTPUT_DIR / 'test_predictions'  
PRED_DIR.mkdir(exist_ok=True)

print('='*70)
print('LIGHTGBM EVALUATION')
print('='*70)
print()

for config_label, cfg in CONFIGS.items():
    print(f'\n{"─"*60}')
    print(f'Config: {config_label}')
    print(f'{"─"*60}')

    config_name = cfg['lgbm_config']
    target      = cfg['target']
    features    = cfg['features']


    # Create targets for this config
    # Each config has its own target variable:
    #  Sub-Enhanced --> mhw_sst
    #   0-50m -->  mhw_layer1  |  50-100m--> mhw_layer2
    #   100-150m --> mhw_layer3
    # create_multi_output_targets() creates ALL lead columns
    # at once and drops rows where ANY lead target is NaN.
    # Identical to training pipeline
    print(f'  Step 1: Creating targets (target={target})...')

    train_df_t, target_cols = create_multi_output_targets(
        df_train_raw, target, LEAD_TIMES
    )
    cal_df_t,  _ = create_multi_output_targets(
        df_cal_raw, target, LEAD_TIMES
    )
    thr_df_t,  _ = create_multi_output_targets(
        df_thr_raw, target, LEAD_TIMES
    )
    test_df_t, _ = create_multi_output_targets(
        df_test_raw, target, LEAD_TIMES
    )

    # Create lag features for LightGBM
    # lag0 = today, lag1 = yesterday, -->, lag14
    # Only all-NaN rows dropped (coastal NaN kept)
    print(f'  Step 2: Creating lag features '
          f'({len(features)} features × 15 lags)...')

    train_lagged, lag_cols = create_lag_features(
        train_df_t, features, n_lags=14
    )
    cal_lagged,  _ = create_lag_features(cal_df_t,  features, n_lags=14)
    thr_lagged,  _ = create_lag_features(thr_df_t,  features, n_lags=14)
    test_lagged, _ = create_lag_features(test_df_t, features, n_lags=14)

    # Get lag columns for THIS config's features only
    # (SST-only has 8×15=120, depth models have 11×15=165)
    # lag0 through lag14 are included (range 0-14)
    config_lag_cols = [
        c for c in lag_cols
        if any(c.startswith(f + '_lag') for f in features)
    ]
    print(f'  Lag features: {len(config_lag_cols)}')

    for lead in LEAD_TIMES:
        print(f'\n  ▸ Lead {lead}d')
        target_col = f'{target}_lead{lead}'

        #  Load saved LightGBM model
        # Each config has its own subfolder: lgbm/{layer_folder}/
        model_path = cfg['lgbm_dir'] / \
            f'lgbm_{config_name}_lead{lead}.pkl'
        if not model_path.exists():
            print(f'    Not found: {model_path.name} — skipping')
            continue
        with open(model_path, 'rb') as f:
            model = pickle.load(f)

        # best_iteration from LightGBM early stopping
        best_epoch_lgbm = getattr(model, 'best_iteration_', None)

        #  Prepare data per split
        # target_col created by create_multi_output_targets() above
        # lag columns created by create_lag_features() above
        cal_df  = cal_lagged
        thr_df  = thr_lagged
        test_df = test_lagged

        X_cal  = cal_df[config_lag_cols].values.astype(np.float32)
        y_cal  = cal_df[target_col].values.astype(np.int8)
        X_thr  = thr_df[config_lag_cols].values.astype(np.float32)
        y_thr  = thr_df[target_col].values.astype(np.int8)
        X_test = test_df[config_lag_cols].values.astype(np.float32)
        y_test = test_df[target_col].values.astype(np.int8)

        #  DYNAMIC PI RATES
        # Computed from ACTUAL data for THIS config+lead
        # NOT hardcoded as each layer/lead has different MHW rate.
        pi_train, pi_cal_v, pi_thr_v, pi_test_v = compute_pi_rates(
            train_lagged, cal_df, thr_df, test_df, target_col
        )

        print(f'    Samples: cal={len(y_cal):,}  '
              f'thr={len(y_thr):,}  '
              f'test={len(y_test):,}')
        print(f'    MHW rates (dynamic):')
        print(f'      train={pi_train*100:.1f}%  '
              f'cal={pi_cal_v*100:.1f}%  '
              f'thr={pi_thr_v*100:.1f}%  '
              f'test={pi_test_v*100:.1f}%')

        # Raw probabilities
        # LightGBM predict_proba returns [P(0), P(1)]
        # We need P(1) = probability of MHW
        raw_cal_p  = model.predict_proba(X_cal)[:, 1]
        raw_thr_p  = model.predict_proba(X_thr)[:, 1]
        raw_test_p = model.predict_proba(X_test)[:, 1]

        # Beta calibration on CAL split
        # Fitted ONLY on cal split — not on thr or test!
        beta_cal, cal_method = fit_lgbm_calibrator(
            raw_cal_p, y_cal
        )
        cal_cal_p  = apply_calibrator(beta_cal, raw_cal_p)
        cal_thr_p  = apply_calibrator(beta_cal, raw_thr_p)
        cal_test_p = apply_calibrator(beta_cal, raw_test_p)

        # Check if calibration helped on CAL split
        b_raw = float(brier_score_loss(y_cal, raw_cal_p))
        b_cal = float(brier_score_loss(y_cal, cal_cal_p))
        print(f'    {cal_method} Brier on CAL: '
              f'{b_raw:.4f} → {b_cal:.4f}  '
              f'{"improved" if b_cal < b_raw else " worse — use raw for this model"}')

        # Save calibrator only if it improved
        if b_cal < b_raw:
            with open(OUTPUT_DIR / f'lgbm_{config_name}_lead{lead}_calibrator.pkl', 'wb') as f:
                pickle.dump(beta_cal, f)

        else:
            # Calibration made things worse — save flag so dashboard uses raw probs
            print(f'    Skipping calibrator save — raw probabilities preferred')


        # SLD prior correction
        # CHANGED AFTER PEER REVIEW: the target prior is the THR-split
        # MHW rate (pi_thr_v), NOT the test rate. The test rate comes from
        # test labels and must not be used to build predictions.
        sld_thr_p  = sld_prior_correction(
            cal_thr_p, pi_cal_v, pi_thr_v
        )
        sld_test_p = sld_prior_correction(
            cal_test_p, pi_cal_v, pi_thr_v
        )


        # Tune thresholds and evaluate
        rows = evaluate_all_calibrations(
            y_thr, raw_thr_p, cal_thr_p, sld_thr_p,
            y_test, raw_test_p, cal_test_p, sld_test_p,
            'LightGBM', config_label, target, lead,
            cal_method,
            best_epoch       = best_epoch_lgbm,
            best_val_auc     = None,
            brier_raw        = b_raw,
            brier_calibrated = b_cal,
            pi_train         = pi_train,
            pi_cal           = pi_cal_v,
            pi_thr           = pi_thr_v,
            pi_test          = pi_test_v,
        )

        # ADDED: THR-split selection scores + onset recall
        y_now = np.nan_to_num(
            test_df[target].values.astype(float), nan=0.0
        ).astype(np.int8)                   # MHW state on day t
        sel = selection_scores(y_thr, {
            'none':              raw_thr_p,
            cal_method:          cal_thr_p,
            f'{cal_method}+SLD': sld_thr_p,
        })
        rows = add_selection_and_onset(rows, sel, {
            'none':              raw_test_p,
            cal_method:          cal_test_p,
            f'{cal_method}+SLD': sld_test_p,
        }, y_test, y_now)
        all_results.extend(rows)

        # ADDED: persistence baseline on the same test rows
        all_results.append(persistence_row(
            y_test, y_now, 'LightGBM', config_label, target, lead,
            pi_train=pi_train, pi_cal=pi_cal_v,
            pi_thr=pi_thr_v, pi_test=pi_test_v,
        ))

        # ADDED: common cells (same predictions, same thresholds)
        probs_test = {
            'none':              raw_test_p,
            cal_method:          cal_test_p,
            f'{cal_method}+SLD': sld_test_p,
        }
        locs   = test_df['location'].values
        common = np.isin(locs, list(COMMON_LOCS))
        inC = lambda d: d[d['location'].isin(COMMON_LOCS)]
        pc = compute_pi_rates(inC(train_lagged), inC(cal_df),
                              inC(thr_df), inC(test_df), target_col)
        common_results.extend(common_cell_rows(
            rows, probs_test, y_test, y_now, common,
            dict(pi_train=pc[0], pi_cal=pc[1], pi_thr=pc[2], pi_test=pc[3]),
            'LightGBM', config_label, target, lead))

        # ADDED: save per-sample test predictions 
        save_test_predictions(
            PRED_DIR / f'LightGBM_{config_label}_lead{lead}.parquet',
            locs, test_df['time'].values, y_test, y_now, probs_test)


        #  Reliability diagram
        plot_reliability_diagram(
            y_test,
            {
                'Raw':              raw_test_p,
                cal_method:         cal_test_p,
                f'{cal_method}+SLD': sld_test_p,
            },
            config=config_label, lead=lead,
            save_path=OUTPUT_DIR /
                f'lgbm_{config_name}_lead{lead}_reliability.png'
        )

        del model; gc.collect()


    # Clean up this config's data before next config
    # Prevents memory accumulation across 4 configs
    del train_lagged, cal_lagged, thr_lagged, test_lagged
    del train_df_t, cal_df_t, thr_df_t, test_df_t
    gc.collect()
lgbm_result_count = len(all_results)
print(f'\nLightGBM evaluation complete')
print(f'   Rows added: {lgbm_result_count:,}')
print(f'   Expected: configs × 6 leads × 7 rows '
      f'(6 model rows + 1 persistence row)')


Output hidden; open in https://colab.research.google.com to view.

## Evaluate all LSTM models

In [ ]:

print('='*70)
print('LSTM EVALUATION')
print('='*70)

for config_label, cfg in CONFIGS.items():
    print(f'\n{"─"*60}')
    print(f'Config: {config_label}')
    print(f'{"─"*60}')

    config_name = cfg['lstm_config']
    lstm_dir    = cfg['lstm_dir']    # layer-specific subfolder
    target      = cfg['target']
    features    = cfg['features']


    #  Create targets for this config
    # LSTM also needs target columns in the dataframe.
    # MaskedLSTMDataset reads target_col from the df directly.
    # No lag features needed — LSTM window provides temporal context.
    print(f'  Creating targets (target={target})...')

    train_df_t, target_cols = create_multi_output_targets(
        df_train_raw, target, LEAD_TIMES
    )
    cal_df_t,  _ = create_multi_output_targets(
        df_cal_raw, target, LEAD_TIMES
    )
    thr_df_t,  _ = create_multi_output_targets(
        df_thr_raw, target, LEAD_TIMES
    )
    test_df_t, _ = create_multi_output_targets(
        df_test_raw, target, LEAD_TIMES
    )

    for lead in LEAD_TIMES:
        print(f'\n --> Lead {lead}d')
        target_col = f'{target}_lead{lead}'

        #  Read per-split dataframes
        # target_col already exists in each df
        # created by create_multi_output_targets() above
        cal_df     = cal_df_t
        thr_df     = thr_df_t
        test_df    = test_df_t
        train_df_l = train_df_t   # for pi_train

        # Load model + scaler + training history
        model_path   = Path(lstm_dir) / \
            f'lstm_{config_name}_lead{lead}_model.keras'
        scaler_path  = Path(lstm_dir) / \
            f'lstm_{config_name}_lead{lead}_scaler.pkl'
        history_path = Path(lstm_dir) / \
            f'lstm_{config_name}_lead{lead}_history.pkl'

        if not model_path.exists():
            print(f'  Not found: {model_path.name} — skipping')
            continue

        # Load model WITHOUT compiling
        # (we only need forward pass, not loss function)
        model = tf.keras.models.load_model(
            model_path, compile=False
        )
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)

        # Load training history for best_epoch, best_val_auc
        best_epoch_lstm   = None
        best_val_auc_lstm = None
        if history_path.exists():
            with open(history_path, 'rb') as f:
                hist = pickle.load(f)
            if 'val_auc' in hist:
                best_epoch_lstm   = int(
                    np.argmax(hist['val_auc'])
                ) + 1
                best_val_auc_lstm = float(max(hist['val_auc']))
            print(f'    Best epoch: {best_epoch_lstm}  '
                  f'Best val AUC: {best_val_auc_lstm:.3f}')

        # Build datasets
        cal_ds  = MaskedLSTMDataset(
            cal_df, features, target_col,
            WINDOW_SIZE, BATCH_SIZE, scaler, MASK_VALUE
        )
        thr_ds  = MaskedLSTMDataset(
            thr_df, features, target_col,
            WINDOW_SIZE, BATCH_SIZE, scaler, MASK_VALUE
        )
        test_ds = MaskedLSTMDataset(
            test_df, features, target_col,
            WINDOW_SIZE, BATCH_SIZE, scaler, MASK_VALUE
        )

        # Check all datasets have sequences
        if any(d.n_sequences == 0
               for d in [cal_ds, thr_ds, test_ds]):
            print(f'  Empty dataset — skipping')
            continue

        # Get true labels directly from dataset
        # (no batch loop needed)
        y_cal_l  = cal_ds.get_labels()
        y_thr_l  = thr_ds.get_labels()
        y_test_l = test_ds.get_labels()

        # DYNAMIC PI RATES
        # Computed from ACTUAL sequence labels for THIS config+lead
        # Different for each config+lead combination.
        pi_train_l = float(train_df_l[target_col].dropna().mean())
        pi_cal_l   = float(y_cal_l.mean())
        pi_thr_l   = float(y_thr_l.mean())
        pi_test_l  = float(y_test_l.mean())

        print(f'    Sequences: cal={cal_ds.n_sequences:,}  '
              f'thr={thr_ds.n_sequences:,}  '
              f'test={test_ds.n_sequences:,}')
        print(f'    MHW rates (dynamic):')
        print(f'      train={pi_train_l*100:.1f}%  '
              f'cal={pi_cal_l*100:.1f}%  '
              f'thr={pi_thr_l*100:.1f}%  '
              f'test={pi_test_l*100:.1f}%')

        #  Raw sigmoid probabilities
        raw_cal_p  = model.predict(cal_ds,  verbose=0).flatten()
        raw_thr_p  = model.predict(thr_ds,  verbose=0).flatten()
        raw_test_p = model.predict(test_ds, verbose=0).flatten()

        # Extract logits for temperature scaling ─
        # Must use logits (pre-sigmoid) not probabilities
        # Temperature scaling is theoretically optimal for focal loss when applied to logits


        # Replace get_lstm_logits calls:
        logit_cal  = prob_to_logit(raw_cal_p)
        logit_thr  = prob_to_logit(raw_thr_p)
        logit_test = prob_to_logit(raw_test_p)

        # Temperature scaling on CAL logits
        # Fitted on CAL only — not on thr or test.
        ts = TemperatureScaler()
        ts.fit(logit_cal, y_cal_l.astype(np.float32))
        print(f'    Temperature T={ts.T:.4f}  '
              f'(T>1 = focal loss under-confidence corrected)')

        ts_cal_p  = ts.predict_proba(logit_cal)
        ts_thr_p  = ts.predict_proba(logit_thr)
        ts_test_p = ts.predict_proba(logit_test)

        b_raw = float(brier_score_loss(y_cal_l, raw_cal_p))
        b_ts  = float(brier_score_loss(y_cal_l, ts_cal_p))
        print(f'    TempScale Brier on CAL: '
              f'{b_raw:.4f} → {b_ts:.4f}  '
              f'{"improved" if b_ts < b_raw else "worse (T≈1, minimal focal miscal)"}')


        # Save temperature scaler only if it improved
        if b_ts < b_raw:
            with open(OUTPUT_DIR / f'lstm_{config_name}_lead{lead}_calibrator.pkl', 'wb') as f:
                pickle.dump(ts, f)
        else:
            print(f'    Skipping calibrator save — raw probabilities preferred')

        # SLD prior correction
        # CHANGED AFTER PEER REVIEW: the target prior is the THR-split
        # MHW rate (pi_thr_l), NOT the test rate (which comes from test labels).
        sld_thr_p  = sld_prior_correction(
            ts_thr_p, pi_cal_l, pi_thr_l
        )
        sld_test_p = sld_prior_correction(
            ts_test_p, pi_cal_l, pi_thr_l
        )

        # Tune thresholds + evaluate
        rows = evaluate_all_calibrations(
            y_thr_l, raw_thr_p, ts_thr_p, sld_thr_p,
            y_test_l, raw_test_p, ts_test_p, sld_test_p,
            'LSTM', config_label, target, lead,
            'TempScale',
            best_epoch       = best_epoch_lstm,
            best_val_auc     = best_val_auc_lstm,
            brier_raw        = b_raw,
            brier_calibrated = b_ts,
            pi_train         = pi_train_l,
            pi_cal           = pi_cal_l,
            pi_thr           = pi_thr_l,
            pi_test          = pi_test_l,
        )

        # ADDED: THR-split selection scores + onset recall
        y_now_l = lstm_state_at_t(test_df, target, test_ds)  # state on day t
        assert len(y_now_l) == len(y_test_l)
        sel = selection_scores(y_thr_l, {
            'none':          raw_thr_p,
            'TempScale':     ts_thr_p,
            'TempScale+SLD': sld_thr_p,
        })
        rows = add_selection_and_onset(rows, sel, {
            'none':          raw_test_p,
            'TempScale':     ts_test_p,
            'TempScale+SLD': sld_test_p,
        }, y_test_l, y_now_l)
        all_results.extend(rows)

        # ADDED: persistence baseline on the same test rows
        all_results.append(persistence_row(
            y_test_l, y_now_l, 'LSTM', config_label, target, lead,
            pi_train=pi_train_l, pi_cal=pi_cal_l,
            pi_thr=pi_thr_l, pi_test=pi_test_l,
        ))

        # ADDED: common cells (same predictions, same thresholds)
        probs_test = {
            'none':          raw_test_p,
            'TempScale':     ts_test_p,
            'TempScale+SLD': sld_test_p,
        }
        meta   = lstm_cols_at_t(test_df, ['location', 'time'], test_ds)
        common = np.isin(meta['location'], list(COMMON_LOCS))
        inC = lambda d: d[d['location'].isin(COMMON_LOCS)]
        pc = compute_pi_rates(inC(train_df_l), inC(cal_df),
                              inC(thr_df), inC(test_df), target_col)
        common_results.extend(common_cell_rows(
            rows, probs_test, y_test_l, y_now_l, common,
            dict(pi_train=pc[0], pi_cal=pc[1], pi_thr=pc[2], pi_test=pc[3]),
            'LSTM', config_label, target, lead))

        # ADDED: save per-sample test predictions 
        save_test_predictions(
            PRED_DIR / f'LSTM_{config_label}_lead{lead}.parquet',
            meta['location'], meta['time'], y_test_l, y_now_l, probs_test)

        #  Reliability diagram
        plot_reliability_diagram(
            y_test_l,
            {
                'Raw sigmoid':   raw_test_p,
                'TempScale':     ts_test_p,
                'TempScale+SLD': sld_test_p,
            },
            config=config_label, lead=lead,
            save_path=OUTPUT_DIR /
                f'lstm_{config_name}_lead{lead}_reliability.png'
        )

        del model, cal_ds, thr_ds, test_ds
        keras.backend.clear_session()
        gc.collect()


    # Clean up this config's data before next config
    del train_df_t, cal_df_t, thr_df_t, test_df_t
    gc.collect()
lstm_result_count = len(all_results) - lgbm_result_count
print(f'\nLSTM evaluation complete')
print(f'   Rows added: {lstm_result_count:,}')
print(f'   Expected: 4 configs × 6 leads × 6 rows = {4*6*6}')



Output hidden; open in https://colab.research.google.com to view.

## Summary

In [ ]:

results_df = pd.DataFrame(all_results)

# Save full results CSV
results_df.to_csv(
    OUTPUT_DIR / 'all_evaluation_results.csv',
    index=False
)
print(f'Saved: all_evaluation_results.csv')
print(f'   Total rows:   {len(results_df):,}')
print(f'   Columns:      {len(results_df.columns)}')
print(f'   Models:       {results_df["model"].unique()}')
print(f'   Configs:      {results_df["config"].unique()}')
print(f'   Calibrations: {results_df["calibration"].unique()}')

#  Brier score comparison
print('\n' + '='*70)
print('BRIER SCORE —(lower = better)')
print('Rows = model+config+lead, Columns = calibration method')
print('='*70)

brier_pivot = results_df[
    results_df['threshold_strategy'] == 'mhw_f1'
].groupby(
    ['model', 'config', 'lead_time', 'calibration']
)['brier'].mean().round(4).unstack('calibration')

print(brier_pivot.to_string())
brier_pivot.to_csv(OUTPUT_DIR / 'brier_comparison.csv')
print(f'Saved: brier_comparison.csv')

# Best calibration per model+config+lead
# CHANGED AFTER PEER REVIEW:
#   The variant is chosen by Brier rank + ECE rank on the THR split
#   (columns brier_thr, ece_thr). Test-set scores are NOT used to choose.
#   Only the chosen variant's TEST metrics are reported.
print('\n' + '='*70)
print('BEST CALIBRATION (lowest THR-split Brier rank + ECE rank)')
print('='*70)

model_rows = results_df[results_df['threshold_strategy'] != 'persistence']
mhw_df_summary = model_rows[model_rows['threshold_strategy'] == 'mhw_f1']

best_rows = []
for (model_n, config, lead), grp in mhw_df_summary.groupby(
    ['model', 'config', 'lead_time']
):
    grp = grp.copy()
    grp['brier_rank'] = grp['brier_thr'].rank()
    grp['ece_rank']   = grp['ece_thr'].rank()
    grp['combined']   = grp['brier_rank'] + grp['ece_rank']
    best_rows.append(grp.loc[grp['combined'].idxmin()])

best_df = pd.DataFrame(best_rows)
assert best_df['brier_thr'].notna().all()
best_df.to_csv(OUTPUT_DIR / 'best_per_model.csv', index=False)
print('Saved: best_per_model.csv (variant chosen on THR split)')

# Persistence baseline and skill relative to it
pers_df = results_df[results_df['threshold_strategy'] == 'persistence'].copy()
pers_df.to_csv(OUTPUT_DIR / 'persistence_baseline.csv', index=False)
print(f'Saved: persistence_baseline.csv ({len(pers_df)} rows)')

pers_df['model_rows'] = pers_df['model'].str.extract(r'\((\w+) rows\)')[0]
cmp = best_df.merge(
    pers_df[['model_rows', 'config', 'lead_time',
             'auc', 'sedi', 'bss', 'brier', 'onset_tpr']],
    left_on=['model', 'config', 'lead_time'],
    right_on=['model_rows', 'config', 'lead_time'],
    suffixes=('', '_pers'),
)
for m in ['auc', 'sedi', 'bss', 'brier']:
    cmp[f'{m}_minus_pers'] = cmp[m] - cmp[f'{m}_pers']
cmp.to_csv(OUTPUT_DIR / 'model_vs_persistence.csv', index=False)
print('Saved: model_vs_persistence.csv')
print(cmp[['model', 'config', 'lead_time', 'calibration',
           'auc', 'auc_pers', 'sedi', 'sedi_pers',
           'bss', 'bss_pers', 'onset_tpr']]
      .sort_values(['config', 'model', 'lead_time'])
      .round(3).to_string(index=False))

display_cols = [
    'model', 'config', 'lead_time', 'calibration',
    'auc', 'brier', 'bss', 'sedi',
    'f1', 'macro_f1', 'tpr', 'fpr', 'csi',
    'pi_test', 'brier_thr', 'ece_thr', 'onset_tpr'
]
avail = [c for c in display_cols if c in best_df.columns]
print(best_df[avail].sort_values(
    ['config', 'model', 'lead_time']
).round(3).to_string(index=False))

#  Overall summary statistics
print('\n' + '='*70)
print('OVERALL SUMMARY — mean metrics per model type × calibration')
print('='*70)

for model_name in ['LightGBM', 'LSTM']:
    for cal in ['none', 'Beta', 'Beta+SLD',
                'TempScale', 'TempScale+SLD',
                'Isotonic', 'Isotonic+SLD']:
        sub = results_df[
            (results_df['model'] == model_name) &
            (results_df['calibration'] == cal) &
            (results_df['threshold_strategy'] == 'mhw_f1')
        ]
        if len(sub) == 0:
            continue
        print(f'{model_name:10s} | {cal:17s} | '
              f'AUC={sub["auc"].mean():.3f}  '
              f'F1={sub["f1"].mean():.3f}  '
              f'BSS={sub["bss"].mean():.3f}  '
              f'SEDI={sub["sedi"].mean():.3f}  '
              f'Brier={sub["brier"].mean():.4f}')

# ADDED: common-cell results (same locations for every depth layer)
common_df = pd.DataFrame(common_results)
common_df.to_csv(OUTPUT_DIR / 'common_cells_results.csv', index=False)
chosen = best_df[['model', 'config', 'lead_time', 'calibration']]
common_chosen = pd.concat([
    common_df.merge(chosen, on=['model', 'config', 'lead_time', 'calibration']),
    common_df[common_df['threshold_strategy'] == 'persistence'],
])
common_chosen.to_csv(OUTPUT_DIR / 'common_cells_chosen.csv', index=False)
print('\n' + '='*70)
print('COMMON CELLS (seafloor > 150 m) — chosen variant + persistence')
print('='*70)
print(common_chosen[['model', 'config', 'lead_time', 'calibration',
                     'auc', 'sedi', 'bss', 'pi_train', 'pi_cal',
                     'pi_thr', 'pi_test', 'n_test']]
      .sort_values(['config', 'model', 'lead_time'])
      .round(3).to_string(index=False))

print(f'\n All files saved to: {OUTPUT_DIR}')
print('\nOutput files:')
print('  all_evaluation_results.csv  — model rows + persistence rows')
print('  brier_comparison.csv        — calibration comparison table')
print('  best_per_model.csv          — variant chosen on THR split')
print('  persistence_baseline.csv    — persistence scores per config/lead')
print('  model_vs_persistence.csv    — model minus persistence')
print('  common_cells_results.csv    — all variants on common cells')
print('  common_cells_chosen.csv     — chosen variant + persistence, common cells')
print('  test_predictions/*.parquet  — per-sample predictions')
print('  *_reliability.png           — reliability diagrams')
print('\nNext steps:')
print('  1. brier_comparison.csv:')
print('     decides if calibration method improved each model')
print('     If Beta > none --> use calibrated probs in dashboard')
print('     If Beta < none --> calibration hurt, use raw probs')
print('  2. best_per_model.csv:')
print('     best model+config per lead time')
print('     this to decides dashboard model selection')
print('  3. all_evaluation_results.csv:')
print('     Full results for thesis tables and figures')
print('     Compare LightGBM vs LSTM across all conditions')

Saved: all_evaluation_results.csv
   Total rows:   336
   Columns:      53
   Models:       ['LightGBM' 'Persistence (LightGBM rows)' 'LSTM' 'Persistence (LSTM rows)']
   Configs:      ['Sub-SST' '0-50m' '50-100m' '100-150m']
   Calibrations: ['none' 'Beta' 'Beta+SLD' 'persistence' 'TempScale' 'TempScale+SLD']

BRIER SCORE —(lower = better)
Rows = model+config+lead, Columns = calibration method
calibration                    Beta  Beta+SLD  TempScale  TempScale+SLD    none
model    config   lead_time                                                    
LSTM     0-50m    3             NaN       NaN     0.1917         0.2368  0.1951
                  5             NaN       NaN     0.1921         0.2335  0.1950
                  7             NaN       NaN     0.2045         0.2508  0.2045
                  9             NaN       NaN     0.2136         0.2651  0.2111
                  12            NaN       NaN     0.2103         0.2582  0.2098
                  14            NaN       